# Stage 5 — Random Forest on Stage 3 Latent Vectors (Colab Ready)

This notebook:
- downloads **Stage 2 v2** outputs for labels / weights,
- downloads **Stage 3** VAE latent vectors,
- trains **Random Forest** on the 8D latent space,
- evaluates on the full test set,
- saves the trained model and predictions.

> Designed to run in Google Colab.

In [ ]:
!pip install -q kaggle kagglehub pandas pyarrow scikit-learn

In [ ]:
# ============================================================
# Kaggle Setup (Supports MULTIPLE API KEYS)
# ============================================================
import os, json, kagglehub

def set_kaggle_credentials(username, key):
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "w") as f:
        json.dump({"username": username, "key": key}, f)
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print(f"Switched to Kaggle account: {username}")

# ============================================================
# 🔹 STEP 1: Stage 2 Dataset (PASTE KEY HERE)
# ============================================================

STAGE2_USERNAME = "monadarling143"
STAGE2_KEY      = "8579619194140e89af885820ef6bdb48"

set_kaggle_credentials(STAGE2_USERNAME, STAGE2_KEY)

stage2_path = kagglehub.dataset_download("monadarling143/stage-2-output-v2")
print("Stage 2 downloaded at:", stage2_path)


# ============================================================
# 🔹 STEP 2: Stage 3 Dataset (PASTE KEY HERE)
# ============================================================

STAGE3_USERNAME = "gandramonishreddy"
STAGE3_KEY      = "86cb782ce481b01b5994fc903cd75f61"

set_kaggle_credentials(STAGE3_USERNAME, STAGE3_KEY)

stage3_path = kagglehub.dataset_download("gandramonishreddy/stage-3-vae-outputs")
print("Stage 3 downloaded at:", stage3_path)

Switched to Kaggle account: monadarling143


100%|██████████| 1.66G/1.66G [00:28<00:00, 63.4MB/s]

Extracting files...


Stage 2 downloaded at: /root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1
Switched to Kaggle account: gandramonishreddy


100%|██████████| 200M/200M [00:02<00:00, 87.3MB/s]

Extracting files...


Stage 3 downloaded at: /root/.cache/kagglehub/datasets/gandramonishreddy/stage-3-vae-outputs/versions/1


In [ ]:
import os
import json
import time
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

SEED = 42
CLASS_NAMES = ["NORMALL", "DoSD", "PROBE", "EXPLOIT", "MALWARE"]
N_CLASSES = 5

STAGE5_DIR = "/content/stage5_outputs"
os.makedirs(STAGE5_DIR, exist_ok=True)

print("Imports ready.")

Imports ready.


In [ ]:
# Load Stage 3 latent vectors
z_train_df = pd.read_parquet(f"{stage3_path}/vae_a_output_16/vae_a_z_train.parquet")
z_test_df  = pd.read_parquet(f"{stage3_path}/vae_a_output_16/vae_a_z_test.parquet")

Z_TRAIN = z_train_df.values.astype(np.float32)
Z_TEST  = z_test_df.values.astype(np.float32)

print("Z_TRAIN:", Z_TRAIN.shape, Z_TRAIN.dtype)
print("Z_TEST :", Z_TEST.shape, Z_TEST.dtype)

# Load Stage 2 labels
STAGE2_SUBFOLDER = "STAGE_2_OUTPUT/stage_2_with_zero_v2"
y_train_df = pd.read_parquet(f"{stage2_path}/{STAGE2_SUBFOLDER}/stage2_y_train.parquet")
y_test_df  = pd.read_parquet(f"{stage2_path}/{STAGE2_SUBFOLDER}/stage2_y_test.parquet")

Y_TRAIN = y_train_df.iloc[:, 0].values.astype(int)
Y_TEST  = y_test_df.iloc[:, 0].values.astype(int)

print("Y_TRAIN:", Y_TRAIN.shape, Y_TRAIN.dtype)
print("Y_TEST :", Y_TEST.shape, Y_TEST.dtype)

# Load class weights and feature names
with open(f"{stage2_path}/{STAGE2_SUBFOLDER}/stage2_class_weights.json", "r") as f:
    cw_data = json.load(f)

CLASS_WEIGHTS = {int(k): float(v) for k, v in cw_data["class_weights"].items()}
print("Class weights loaded:", CLASS_WEIGHTS)

feature_names_path = f"{stage2_path}/{STAGE2_SUBFOLDER}/stage2_feature_names.json"
if os.path.exists(feature_names_path):
    with open(feature_names_path, "r") as f:
        FEATURE_NAMES = json.load(f)
    print("Feature names loaded:", len(FEATURE_NAMES))
else:
    FEATURE_NAMES = None
    print("Feature names file not found; continuing without it.")

Z_TRAIN: (2381042, 8) float32
Z_TEST : (573807, 8) float32
Y_TRAIN: (2381042,) int64
Y_TEST : (573807,) int64
Class weights loaded: {0: 0.26263988691572343, 1: 1.5220112438914475, 2: 3.174722666666667, 3: 5.01272, 4: 47.62084}
Feature names loaded: 7


In [ ]:
import os
print(f"Contents of {stage3_path}:")
for root, dirs, files in os.walk(stage3_path):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

Contents of /root/.cache/kagglehub/datasets/gandramonishreddy/stage-3-vae-outputs/versions/1:
/root/.cache/kagglehub/datasets/gandramonishreddy/stage-3-vae-outputs/versions/1/vae_b_output_16
/root/.cache/kagglehub/datasets/gandramonishreddy/stage-3-vae-outputs/versions/1/vae_a_output_16
/root/.cache/kagglehub/datasets/gandramonishreddy/stage-3-vae-outputs/versions/1/vae_b_output_16/vae-b_angle_distributions.png
/root/.cache/kagglehub/datasets/gandramonishreddy/stage-3-vae-outputs/versions/1/vae_b_output_16/vae_b_z_test.parquet
/root/.cache/kagglehub/datasets/gandramonishreddy/stage-3-vae-outputs/versions/1/vae_b_output_16/vae_b_best.pt
/root/.cache/kagglehub/datasets/gandramonishreddy/stage-3-vae-outputs/versions/1/vae_b_output_16/vae-b_training_curves.png
/root/.cache/kagglehub/datasets/gandramonishreddy/stage-3-vae-outputs/versions/1/vae_b_output_16/vae-b_tsne.png
/root/.cache/kagglehub/datasets/gandramonishreddy/stage-3-vae-outputs/versions/1/vae_b_output_16/vae_b_z_train.parquet
/r

In [ ]:
# Sanity checks
assert Z_TRAIN.shape[0] == Y_TRAIN.shape[0], f"Train mismatch: {Z_TRAIN.shape[0]} vs {Y_TRAIN.shape[0]}"
assert Z_TEST.shape[0] == Y_TEST.shape[0], f"Test mismatch: {Z_TEST.shape[0]} vs {Y_TEST.shape[0]}"
assert Z_TRAIN.shape[1] == 8, f"Expected 8 latent dims, got {Z_TRAIN.shape[1]}"
assert len(CLASS_WEIGHTS) == 5, f"Expected 5 class weights, got {len(CLASS_WEIGHTS)}"

SAMPLE_WEIGHTS = np.array([CLASS_WEIGHTS[y] for y in Y_TRAIN], dtype=np.float32)

print("All shapes verified.")
print("Sample weight stats:")
print("  min :", float(SAMPLE_WEIGHTS.min()))
print("  max :", float(SAMPLE_WEIGHTS.max()))
print("  mean:", float(SAMPLE_WEIGHTS.mean()))

All shapes verified.
Sample weight stats:
  min : 0.2626398801803589
  max : 47.6208381652832
  mean: 1.000000238418579


In [ ]:
import os
print(f"Contents of {stage2_path}:")
for root, dirs, files in os.walk(stage2_path):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

Contents of /root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1:
/root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1/STAGE_2_OUTPUT
/root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1/STAGE_2_OUTPUT/stage_2_with_zero_v2
/root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1/STAGE_2_OUTPUT/stage_2_without_zero_v2
/root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1/STAGE_2_OUTPUT/stage_2_with_zero_v2/stage2_y_train.parquet
/root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1/STAGE_2_OUTPUT/stage_2_with_zero_v2/stage2_sentinel_mask_train.parquet
/root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1/STAGE_2_OUTPUT/stage_2_with_zero_v2/stage2_y_test.parquet
/root/.cache/kagglehub/datasets/monadarling143/stage-2-output-v2/versions/1/STAGE_2_OUTPUT/stage_2_with_zero_v2/stage2_preprocessing_artefacts.json
/root/.cache/kagglehub/datase

In [ ]:
# Random Forest training
rf_model = RandomForestClassifier(
    n_estimators=200,          # increase to 500 if you have enough RAM/time
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced_subsample",
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=SEED,
    verbose=1,
)

print("Training Random Forest...")
t0 = time.time()
rf_model.fit(Z_TRAIN, Y_TRAIN)
rf_time = time.time() - t0

print(f"Training complete in {rf_time/60:.2f} minutes")
print(f"OOB score: {rf_model.oob_score_:.4f}")

Training Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  9.8min
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed: 38.3min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed: 39.0min finished


Training complete in 40.06 minutes
OOB score: 0.9737


In [ ]:
# Predictions
RF_PROBA = rf_model.predict_proba(Z_TEST).astype(np.float32)
RF_PRED  = np.argmax(RF_PROBA, axis=1)

print("RF_PROBA:", RF_PROBA.shape)
print("Probability sums (first 5):", RF_PROBA.sum(axis=1)[:5])

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    6.5s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:   16.1s


RF_PROBA: (573807, 5)
Probability sums (first 5): [1. 1. 1. 1. 1.]


[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:   16.4s finished


In [ ]:
# Evaluation
rf_f1_macro = f1_score(Y_TEST, RF_PRED, average="macro", zero_division=0)
rf_f1_weighted = f1_score(Y_TEST, RF_PRED, average="weighted", zero_division=0)
rf_acc = accuracy_score(Y_TEST, RF_PRED)
rf_f1_cls = f1_score(Y_TEST, RF_PRED, average=None, zero_division=0)

print("=" * 70)
print("Random Forest Results")
print("=" * 70)
print(f"Accuracy    : {rf_acc:.4f}")
print(f"F1-macro    : {rf_f1_macro:.4f}")
print(f"F1-weighted : {rf_f1_weighted:.4f}")
print()
print("Per-class F1:")
for name, score in zip(CLASS_NAMES, rf_f1_cls):
    print(f"  {name:<10s}: {score:.4f}")

print("\nClassification report:")
print(classification_report(Y_TEST, RF_PRED, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(Y_TEST, RF_PRED)
print("\nConfusion matrix:")
print(cm)

try:
    auc = roc_auc_score(Y_TEST, RF_PROBA, multi_class="ovr", average="macro")
    print(f"Macro AUC-ROC: {auc:.4f}")
except Exception as e:
    print(f"AUC-ROC skipped: {e}")

Random Forest Results
Accuracy    : 0.9783
F1-macro    : 0.7980
F1-weighted : 0.9802

Per-class F1:
  NORMALL   : 0.9936
  DoSD      : 0.9766
  PROBE     : 0.8905
  EXPLOIT   : 0.7744
  MALWARE   : 0.3550

Classification report:
              precision    recall  f1-score   support

     NORMALL       1.00      0.99      0.99    453290
        DoSD       1.00      0.96      0.98     78221
       PROBE       0.86      0.92      0.89     29137
     EXPLOIT       0.68      0.91      0.77     11966
     MALWARE       0.25      0.60      0.35      1193

    accuracy                           0.98    573807
   macro avg       0.76      0.88      0.80    573807
weighted avg       0.98      0.98      0.98    573807


Confusion matrix:
[[447892    293   4107    698    300]
 [    80  74950     55   2629    507]
 [   184     13  26938   1491    511]
 [    75      7    211  10852    821]
 [    32      0     50    392    719]]
Macro AUC-ROC: 0.9922


In [ ]:
# Save outputs
with open(f"{STAGE5_DIR}/rf_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

pd.DataFrame(
    RF_PROBA,
    columns=[f"p_{c}" for c in CLASS_NAMES]
).to_parquet(f"{STAGE5_DIR}/rf_test_proba.parquet", index=False)

pred_df = pd.DataFrame({
    "y_true": Y_TEST,
    "y_pred_rf": RF_PRED
})
pred_df.to_parquet(f"{STAGE5_DIR}/rf_predictions.parquet", index=False)

results = {
    "accuracy": float(rf_acc),
    "f1_macro": float(rf_f1_macro),
    "f1_weighted": float(rf_f1_weighted),
    "f1_per_class": {name: float(score) for name, score in zip(CLASS_NAMES, rf_f1_cls)},
    "oob_score": float(rf_model.oob_score_),
    "n_train": int(Z_TRAIN.shape[0]),
    "n_test": int(Z_TEST.shape[0]),
    "latent_dim": int(Z_TRAIN.shape[1]),
    "class_names": CLASS_NAMES,
    "class_weights": CLASS_WEIGHTS,
    "training_time_minutes": round(rf_time / 60, 2),
}

with open(f"{STAGE5_DIR}/rf_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved:")
for fname in sorted(os.listdir(STAGE5_DIR)):
    print(" ", fname)

Saved:
  rf_model.pkl
  rf_predictions.parquet
  rf_results.json
  rf_test_proba.parquet


### Optional: if Colab runs out of RAM

Uncomment the following lines before training:

```python
# Z_TRAIN = Z_TRAIN[:500000]
# Y_TRAIN = Y_TRAIN[:500000]
# SAMPLE_WEIGHTS = SAMPLE_WEIGHTS[:500000]
```

For the full research run, keep the complete dataset.

In [ ]:
import shutil
from google.colab import files

output_archive_name = "stage5_outputs.zip"
shutil.make_archive("stage5_outputs", 'zip', STAGE5_DIR)
files.download(output_archive_name)

print(f"'{output_archive_name}' has been created and is ready for download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'stage5_outputs.zip' has been created and is ready for download.
